# 🐙 Day 02a: Docker Compose — Multi-Container Orchestration

---

## 🎯 What You'll Master Today
- docker-compose.yml structure
- Multi-container setup (API + DB + Redis)
- Volumes, networks, and healthchecks
- Development vs production compose files

---

## 🎭 The Orchestra Analogy

Docker is one musician. Docker Compose is the **conductor** — orchestrating multiple musicians (containers) to play together. It defines who plays what, when they start, and how they communicate.

```
╔═══════════════════════════════════════════════════════════════════╗
║  DOCKER COMPOSE — KEY CONCEPTS                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  services:   → Define each container (api, db, redis, worker)   ║
║  volumes:    → Persist data (DB data survives container death)  ║
║  networks:   → Internal communication between services          ║
║  depends_on: → Startup order + health conditions                ║
║  ports:      → Expose to host ("host:container")                ║
║  env_file:   → Load environment variables from .env file        ║
║                                                                   ║
║  Service names ARE hostnames inside the network:                ║
║    api can reach db at: postgres://db:5432/mydb                 ║
║    api can reach redis at: redis://redis:6379                   ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# Development docker-compose.yml
# ============================================================

print("""
# docker-compose.yml (Development)

services:
  api:
    build: .
    command: uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload
    ports:
      - "8000:8000"
    volumes:
      - .:/app                    # Hot-reload: code changes reflect instantly
    env_file:
      - .env
    depends_on:
      db:
        condition: service_healthy  # Wait for DB to be ready
      redis:
        condition: service_started

  db:
    image: postgres:16-alpine
    environment:
      POSTGRES_USER: appuser
      POSTGRES_PASSWORD: apppass
      POSTGRES_DB: mydb
    ports:
      - "5432:5432"               # Expose for dev tools
    volumes:
      - postgres_data:/var/lib/postgresql/data   # Persist data
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U appuser -d mydb"]
      interval: 5s
      timeout: 3s
      retries: 5

  redis:
    image: redis:7-alpine
    ports:
      - "6379:6379"
    volumes:
      - redis_data:/data

volumes:
  postgres_data:
  redis_data:
""")

print("Key dev features:")
print("  --reload      → auto-restart on code changes")
print("  volumes: .:/app → mount code directory")
print("  ports exposed  → connect from dev tools")

In [ ]:
# ============================================================
# Production docker-compose.yml
# ============================================================

print("""
# docker-compose.prod.yml

services:
  api:
    build:
      context: .
      dockerfile: Dockerfile
    command: uvicorn app.main:app --host 0.0.0.0 --port 8000 --workers 4
    ports:
      - "8000:8000"
    env_file:
      - .env.prod
    depends_on:
      db:
        condition: service_healthy
    restart: unless-stopped                 # Auto-restart on crash
    deploy:
      resources:
        limits:
          memory: 512M
          cpus: "0.5"

  db:
    image: postgres:16-alpine
    env_file:
      - .env.prod
    volumes:
      - postgres_data:/var/lib/postgresql/data
    # NO ports exposed — only accessible within network
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U $${POSTGRES_USER} -d $${POSTGRES_DB}"]
      interval: 10s
      timeout: 5s
      retries: 5
    restart: unless-stopped

  redis:
    image: redis:7-alpine
    command: redis-server --maxmemory 256mb --maxmemory-policy allkeys-lru
    volumes:
      - redis_data:/data
    restart: unless-stopped

  worker:
    build: .
    command: celery -A worker.celery_app worker --loglevel=info --concurrency=2
    env_file:
      - .env.prod
    depends_on:
      - redis
      - db
    restart: unless-stopped

  nginx:
    image: nginx:alpine
    ports:
      - "80:80"
      - "443:443"
    volumes:
      - ./nginx.conf:/etc/nginx/conf.d/default.conf
    depends_on:
      - api
    restart: unless-stopped

volumes:
  postgres_data:
  redis_data:
""")

print("Production differences:")
print("  ❌ No --reload, no volume mounts for code")
print("  ❌ No ports exposed for DB/Redis")
print("  ✅ restart: unless-stopped")
print("  ✅ Resource limits")
print("  ✅ Nginx reverse proxy")
print("  ✅ Multiple workers")

In [ ]:
# ============================================================
# Docker Compose Commands
# ============================================================

print("""
Essential commands:

# Start everything
docker compose up -d              # Detached mode
docker compose up --build         # Rebuild images first

# Stop everything
docker compose down               # Stop and remove containers
docker compose down -v            # Also remove volumes (data!)

# View status
docker compose ps                 # Running services
docker compose logs api           # Logs for one service
docker compose logs -f            # Follow all logs

# Execute commands in running container
docker compose exec api /bin/sh   # Shell into api container  
docker compose exec db psql -U appuser -d mydb  # Connect to DB

# Scale (if no host port mapping)
docker compose up --scale worker=3  # 3 Celery workers

# Production with specific file
docker compose -f docker-compose.prod.yml up -d
""")

In [ ]:
# ============================================================
# Volumes Deep Dive
# ============================================================

print("""
Volume types:

1. NAMED VOLUMES — managed by Docker, for persistent data
   volumes:
     postgres_data:/var/lib/postgresql/data
   # Data persists even if container is deleted

2. BIND MOUNTS — map host directory, for development
   volumes:
     .:/app   # Host current dir → container /app
   # Code changes on host reflect in container

3. TMPFS — in-memory only, for secrets/temp data
   tmpfs:
     - /tmp
""")

print("\nNetworks:")
print("  • Compose creates a default network automatically")
print("  • Service names ARE hostnames: api → db (use 'db' as hostname)")
print("  • Custom networks isolate groups of services")

print("\nHealthchecks:")
print("  • condition: service_healthy → wait for health check to pass")
print("  • condition: service_started → just wait for container to start")
print("  • Without condition: depends_on only controls start ORDER")

---

## 🗺️ Docker Compose Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  DOCKER COMPOSE PATTERNS                                          ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  TYPICAL STACK:                                                  ║
║    nginx → api (FastAPI) → db (PostgreSQL) + redis              ║
║                         → worker (Celery)                       ║
║                                                                   ║
║  DEV vs PROD:                                                    ║
║    Dev:  --reload, bind mounts, ports exposed, debug mode       ║
║    Prod: workers, resource limits, no debug, restart policy     ║
║                                                                   ║
║  DATA:                                                           ║
║    Named volumes  → persistent (DB, Redis)                      ║
║    Bind mounts    → development (hot reload)                    ║
║                                                                   ║
║  NETWORKING:                                                     ║
║    Service name = hostname (api, db, redis)                     ║
║    Only expose ports needed externally                          ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Docker Compose vs Docker? | Docker: single container. Compose: multi-container orchestration |
| 2 | How do services communicate? | Default network, service names as hostnames |
| 3 | Named volume vs bind mount? | Named: Docker-managed, persistent. Bind: host dir mapped, for dev |
| 4 | depends_on with healthcheck? | condition: service_healthy ensures dependent is actually ready |
| 5 | How to handle secrets? | env_file, Docker secrets, never in Dockerfile/compose file |
| 6 | Compose vs Kubernetes? | Compose: single host, dev/small prod. K8s: multi-host, large scale |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Compose = multi-container orchestration via YAML     │
## │  • Service names are hostnames on the network           │
## │  • Named volumes persist data across restarts           │
## │  • healthcheck + depends_on for startup ordering        │
## │  • Separate dev and prod compose files                  │
## │  • restart: unless-stopped for production               │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **CI/CD** — Automated test, build, and deploy pipelines